# 06 — Reproducibility & Provenance

**Purpose:** Explain and demonstrate how PBI tracks the identity and provenance of a
data release, so that results can be reproduced and cited correctly.

| | |
|---|---|
| **Expected inputs** | Pipeline logs at `/pipeline-logs` (optional but recommended), `pbi` package installed |
| **Outputs** | Provenance summary saved under `<results>/06_reproducibility/` |

## What you will learn

1. The **three-layer versioning model** used in this project
2. How to read the **pbi package version** programmatically
3. How the **provider schema profile** (`phagescope_v1`) is defined and when it changes
4. How to read and validate **pipeline run provenance** artefacts
5. How to verify the **public data manifest** (snapshot date, checksums)
6. How to inspect **database build metadata**
7. A practical **reproducibility checklist**


---
## Setup


In [ ]:
import sys
import json
import os
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import pbi
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

logs_root = Path('/pipeline-logs')

results_root = Path(os.getenv('PBI_RESULTS_DIR', '/results'))
results_dir = results_root / '06_reproducibility'
results_dir.mkdir(parents=True, exist_ok=True)

print(f'pbi version  : {pbi.__version__}')
print(f'Python       : {sys.version.split()[0]}')
print(f'Results dir  : {results_dir}')


---
## 1. The Three-Layer Versioning Model

PBI deliberately separates three concepts that are often conflated:

```
┌─────────────────────────────────────────────────────────────────────┐
│  Layer 1 — Package version                                          │
│  ─────────────────────────────────────────────────────────────────  │
│  SemVer string in src/pbi/__init__.py   e.g. "0.3.0"               │
│  Read by setup.py; tracked with Git tags (v0.3.0).                  │
│  Bump when the pbi Python API changes.                              │
│                                                                     │
│  Layer 2 — Provider schema profile                                  │
│  ─────────────────────────────────────────────────────────────────  │
│  Stable label in workflow/config/config.yaml                        │
│  public_data_provider.schema_profile   e.g. "phagescope_v1"        │
│  Records the column/semantic contract of the upstream data source.  │
│  Bump to "phagescope_v2" only when the source schema changes.       │
│                                                                     │
│  Layer 3 — Pipeline run provenance                                  │
│  ─────────────────────────────────────────────────────────────────  │
│  JSON artefact written per pipeline execution                       │
│  /pipeline-logs/csv/pipeline_run_provenance.json                    │
│  Records: snapshot_date, git_ref, git_commit, pbi_version, ...      │
└─────────────────────────────────────────────────────────────────────┘
```

**Key principle:** keeping them separate means you can compare two pipeline runs
that used the same schema profile (same data contract) even if the package version
changed between runs.  Conversely, a package update that does not change data
semantics does not force a schema version bump.


---
## 2. Layer 1 — Package Version


In [ ]:
# The single source of truth for the package version is __version__ in
# src/pbi/__init__.py.  setup.py reads it with a regex, so no import is
# needed at build time.

print(f'pbi.__version__ = {repr(pbi.__version__)}')

# Cross-check: read the file line-by-line (avoids loading the full file)
import re
init_py = project_root / 'src' / 'pbi' / '__init__.py'
if init_py.exists():
    file_version = 'not found'
    with init_py.open() as _fh:
        for _line in _fh:
            _m = re.match(r"^__version__\s*=\s*[\"'](.*?)[\"']", _line)
            if _m:
                file_version = _m.group(1)
                break
    match_ok = file_version == pbi.__version__
    print(f'Version in file : {repr(file_version)}  {"✅ match" if match_ok else "❌ mismatch"}')


---
## 3. Layer 2 — Provider Schema Profile

The `schema_profile` field lives in `workflow/config/config.yaml` under
`public_data_provider`.  It is written into every download manifest and into the
DuckDB `dataset_provenance` table during the pipeline run, so it is always
traceable from the data artefacts alone.

Changing this label is a **deliberate, documented decision** — it signals to
downstream consumers that the column layout or data semantics changed.


In [ ]:
# Try to read schema_profile from the workflow config (repo checkout only)
try:
    import yaml  # PyYAML is a pbi dependency
    config_path = project_root / 'workflow' / 'config' / 'config.yaml'
    if config_path.exists():
        with config_path.open() as fh:
            cfg = yaml.safe_load(fh)
        provider = cfg.get('public_data_provider', {})
        print('public_data_provider (from workflow/config/config.yaml):')
        for k, v in provider.items():
            print(f'  {k:<30} {v}')
    else:
        print('workflow/config/config.yaml not found (release package only — see manifest).')
except ImportError:
    print('PyYAML not available; install pyyaml or inspect config.yaml manually.')


---
## 4. Layer 3 — Pipeline Run Provenance

The pipeline writes `/pipeline-logs/csv/pipeline_run_provenance.json` at the
end of a successful run.  Reading it confirms *exactly* which data snapshot,
software version, and (optionally) git commit produced the current database.


In [ ]:
prov_path = logs_root / 'csv' / 'pipeline_run_provenance.json'

if prov_path.exists():
    provenance = json.loads(prov_path.read_text())
    print('pipeline_run_provenance.json')
    print('─' * 55)
    for key, val in provenance.items():
        print(f'  {key:<40} {val}')

    # Save a copy to results for archiving
    out = results_dir / 'pipeline_run_provenance.json'
    out.write_text(json.dumps(provenance, indent=2))
    print(f'\nCopy saved → {out}')
else:
    provenance = {}
    print('ℹ️  pipeline_run_provenance.json not found.')
    print('   Run the Snakemake pipeline to generate this artefact.')


---
## 5. Public Data Manifest

The pipeline also writes `/pipeline-logs/csv/public_data_manifest.json` containing
one entry per downloaded file: URL, HTTP response headers (including `Last-Modified`
and `ETag`), file size, and optionally a checksum.  Together with the provenance
record this allows exact reconstruction of the input data.


In [ ]:
manifest_path = logs_root / 'csv' / 'public_data_manifest.json'

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    # Manifest may be a list or a dict — handle both
    entries = manifest if isinstance(manifest, list) else manifest.get('files', [])
    print(f'Public data manifest — {len(entries)} entries')
    if entries:
        df_manifest = pd.json_normalize(entries[:5])  # preview first 5
        display(df_manifest)
else:
    print('ℹ️  public_data_manifest.json not found.')
    print('   Generated by the download_public_file pipeline step.')


---
## 6. Database Build Metadata

The DuckDB database contains a `dataset_provenance` table (and optionally a
`run_provenance` table) that record the same metadata directly inside the
artefact.  This means the provenance travels with the database file, even when
it is distributed via Zenodo.


In [ ]:
try:
    retriever = pbi.quick_connect()
    conn = retriever.conn

    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchdf()
    prov_tables = [t for t in tables['name'].tolist()
                   if 'provenance' in t.lower() or 'metadata' in t.lower()]

    if prov_tables:
        for tbl in prov_tables:
            print(f'\n── {tbl} ──')
            try:
                display(conn.execute(f'SELECT * FROM {tbl} LIMIT 5').fetchdf())
            except Exception as exc:
                print(f'  Could not query {tbl}: {exc}')
    else:
        print('No provenance tables found in database.')
        print('Available tables:', tables['name'].tolist())

    retriever.close()
except Exception as exc:
    print(f'Could not connect to database: {exc}')
    print('Run the pipeline first or check DATA_PATH.')


---
## 7. Practical Reproducibility Checklist

Use this checklist when sharing or archiving a PBI-derived dataset or analysis.

### Data identity
- [ ] Record `pbi.__version__` in your analysis script / notebook header
- [ ] Record the `schema_profile` from the provenance JSON (`phagescope_v1`, etc.)
- [ ] Record the `snapshot_date` from the provenance JSON
- [ ] Archive `pipeline_run_provenance.json` alongside your results

### Code identity
- [ ] Tag or note the Git commit SHA used to run the pipeline
- [ ] Pin exact dependency versions (`pip freeze > requirements_lock.txt`)

### Data artefacts
- [ ] Include the DuckDB file checksum (or Zenodo DOI) in methods
- [ ] Archive the `public_data_manifest.json` for input traceability

### Quoting in publications

A minimal, reproducible reference sentence:

> Analysis used PBI v{pbi.__version__} (schema profile: `phagescope_v1`, PhageScope
> snapshot {snapshot_date}).  Source code and database available at
> https://github.com/ThibaultSchowing/PBI.

Substitute `{pbi.__version__}` and `{snapshot_date}` from your provenance artefact.


In [ ]:
# Print a ready-to-use citation line from the current provenance
snapshot_date = provenance.get('snapshot_date', '<snapshot_date>') if provenance else '<snapshot_date>'
schema_profile = provenance.get('provider_schema_profile', 'phagescope_v1') if provenance else 'phagescope_v1'

print('Suggested citation line:')
print()
print(f'  PBI v{pbi.__version__} (schema profile: {schema_profile!r},'
      f' PhageScope snapshot {snapshot_date}).')
print('  https://github.com/ThibaultSchowing/PBI')


---
## Summary

| Layer | Artefact | Stable across |
|-------|----------|---------------|
| Package version | `pbi.__version__` in code | software releases |
| Schema profile | `public_data_provider.schema_profile` in config + DB | data schema versions |
| Run provenance | `pipeline_run_provenance.json` + DB tables | individual pipeline runs |

These three layers together give you exact, citable traceability from any analysis
result back to the specific software, data schema, and data snapshot that produced it.
